#Entendendo o problema

A base de dados em questão diz respeito à taxa de churn de clientes em uma empresa de telecomunicações. Mas, afinal, o que é churn?

`Churn`

*  Ocorre quando clientes deixam de fazer negócios com uma empresa
*  Representa a perda de receita
*  Pode ser um indicador de satisfação de clientes em relação aos produtos/serviços.

##Estratégias para reduzir a taxa de churn

*  Melhorar a qualidade de produtos e serviços
*  Oferecer suporte de qualidade a clientes
*  Criar programas de fidelidade

In [1]:
import pandas as pd
import json
import requests
from pydantic import BaseModel, Field, field_validator

class DataLoaderConfig(BaseModel):
    url: str = Field(..., description="URL do arquivo JSON")
    chave: str | None = Field(None, description="Chave do JSON a ser carregada")

    @field_validator("url")
    @classmethod
    def validar_url(cls, v: str) -> str:
        if not (v.startswith("http://") or v.startswith("https://")):
            raise ValueError("URL deve começar com http:// ou https://")
        return v

    model_config = {
        "extra": "allow",
        "json_schema_extra": {
            "example": {
                "url": "https://exemplo.com/dados.json",
                "chave": "dados_churn",
            }
        },
    }

def load_data(config: DataLoaderConfig) -> pd.DataFrame:
    response = requests.get(config.url)
    response.raise_for_status()
    data = response.json()

    # Caso seja um dict com chave específica
    if isinstance(data, dict):
        if config.chave:
            data = data[config.chave]
        else:
            # Sem chave → tenta transformar o dict inteiro
            return pd.DataFrame.from_dict(data)

    # Caso já seja uma lista
    if isinstance(data, list):
        return pd.DataFrame(data)

    # Se não for lista nem dict, encapsula para não quebrar
    return pd.DataFrame([data])

# Exemplo de configs
configs = {
    "churn": DataLoaderConfig(
        url="https://raw.githubusercontent.com/YuriArduino/Estudos_Pandas/refs/heads/data-tests/dataset-telecon.json"
    )
}

# Carregar e visualizar
df = load_data(configs["churn"])
print(df.head())


   id_cliente Churn                                            cliente  \
0  0002-ORFBO   nao  {'genero': 'feminino', 'idoso': 0, 'parceiro':...   
1  0003-MKNFE   nao  {'genero': 'masculino', 'idoso': 0, 'parceiro'...   
2  0004-TLHLJ   sim  {'genero': 'masculino', 'idoso': 0, 'parceiro'...   
3  0011-IGKFF   sim  {'genero': 'masculino', 'idoso': 1, 'parceiro'...   
4  0013-EXCHZ   sim  {'genero': 'feminino', 'idoso': 1, 'parceiro':...   

                                            telefone  \
0  {'servico_telefone': 'sim', 'varias_linhas': '...   
1  {'servico_telefone': 'sim', 'varias_linhas': '...   
2  {'servico_telefone': 'sim', 'varias_linhas': '...   
3  {'servico_telefone': 'sim', 'varias_linhas': '...   
4  {'servico_telefone': 'sim', 'varias_linhas': '...   

                                            internet  \
0  {'servico_internet': 'DSL', 'seguranca_online'...   
1  {'servico_internet': 'DSL', 'seguranca_online'...   
2  {'servico_internet': 'fibra otica', 'seguranca.

In [2]:
dados_churn = df.copy()

###Visualização:

In [3]:
print (dados_churn.shape[1])
print (dados_churn.columns)
print (dados_churn.info())

6
Index(['id_cliente', 'Churn', 'cliente', 'telefone', 'internet', 'conta'], dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7344 entries, 0 to 7343
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id_cliente  7344 non-null   object
 1   Churn       7344 non-null   object
 2   cliente     7344 non-null   object
 3   telefone    7344 non-null   object
 4   internet    7344 non-null   object
 5   conta       7344 non-null   object
dtypes: object(6)
memory usage: 344.4+ KB
None


In [4]:
dados_churn['conta'][0]

{'contrato': None,
 'faturamente_eletronico': None,
 'metodo_pagamento': None,
 'cobranca': {'mensal': None, 'Total': None}}

In [5]:
#Método json_normalize()
pd.json_normalize(dados_churn['conta']).head()

,contrato,faturamente_eletronico,metodo_pagamento,cobranca.mensal,cobranca.Total
0,None,None,None,NaN,None
1,mes a mes,nao,cheque pelo correio,59.9,542.4
2,mes a mes,sim,cheque eletronico,73.9,280.85
3,mes a mes,sim,cheque eletronico,98.0,1237.85
4,mes a mes,sim,cheque pelo correio,83.9,267.4


In [6]:
pd.json_normalize(dados_churn['telefone']).head()

,servico_telefone,varias_linhas
0,sim,nao
1,sim,sim
2,sim,nao
3,sim,nao
4,sim,nao


### Para saber mais: o que são modelos de Machine Learning?

Modelos de *machine learning* permitem que computadores aprendam a realizar tarefas sem precisarem ser programados passo a passo. Eles funcionam a partir de algoritmos que analisam grandes volumes de dados, identificam padrões e constroem representações matemáticas que relacionam entradas (informações fornecidas) a saídas (respostas ou previsões).

🔹 **Exemplo simples**:
Imagine que você queira prever se alguém vai gostar de um filme. O modelo recebe dados como gênero, elenco, diretor(a) e avaliações anteriores, e aprende a reconhecer padrões nesses gostos. Com isso, ele consegue sugerir se uma nova pessoa provavelmente vai gostar ou não de um título específico.

🔹 **Exemplo do dia a dia**:
Nas lojas online, como a Amazon, modelos analisam histórico de compras e navegação para recomendar produtos de interesse — muitas vezes antecipando desejos que nem havíamos pensado.

Esses modelos já estão em todo lugar:

* 📊 detecção de fraudes em transações financeiras,
* 🗣️ reconhecimento de voz,
* 👁️ identificação de imagens,
* 🏥 auxílio em diagnósticos médicos, entre muitos outros.

A grande força do *machine learning* é sua **capacidade de melhorar continuamente**. Quanto mais dados recebem, mais refinados e confiáveis se tornam — conseguindo resolver problemas cada vez mais complexos com rapidez e precisão.

---

### Para saber mais: o que é Churn?

Churn é a taxa de cancelamento de clientes em um determinado período — ou seja, a porcentagem de pessoas que deixam de usar um produto ou serviço.

🔹 **Por que é importante?**
Uma taxa de Churn alta indica que a empresa está perdendo clientes mais rápido do que consegue conquistar novos, sinalizando possíveis problemas com o produto, serviço ou experiência do cliente. Já uma taxa baixa mostra que os clientes estão satisfeitos e fiéis, refletindo saúde e estabilidade para o negócio.

🔹 **Churn e Machine Learning**
Criar modelos de *machine learning* para prever Churn permite identificar clientes em risco de cancelamento antes que isso aconteça. Ao analisar dados históricos, é possível entender padrões de comportamento, antecipar problemas e aplicar ações proativas, como:

* enviar ofertas personalizadas,
* oferecer descontos estratégicos,
* melhorar a experiência do cliente.

Monitorar e reduzir o Churn não só ajuda a manter a base de clientes, mas também fortalece a competitividade e a sustentabilidade do negócio.

---

##Transformando dados em uma tabela

Forma mais pythonica:

*  Em um DataFrame ⇾ normaliza apenas uma coluna

*  Em um objeto JSON ⇾ normaliza todas as colunas aninhadas de uma só vez

`Método with()`

ara utilizar o método with(), na próxima célula, digitaremos with open(""), passando o nome do arquivo dataset-telecon.json, e, em seguida, criaremos uma variável nomeada como f utilizando o termo as.

In [7]:
#exemplo da aula
#with open("dataset-telecon.json") as f:
    #json_bruto = json.load(f)

#meu código via url:
# Já usando seu DataLoaderConfig
config = configs["churn"]

# Carrega o JSON validado
df = load_data(config)

# Se quiser ver o "json_bruto" equivalente
json_bruto = df.to_dict(orient="records")  # transforma de volta em lista de dicts
# Para mostrar as primeiras 5 linhas da lista json_bruto
print(json_bruto[:5])

[{'id_cliente': '0002-ORFBO', 'Churn': 'nao', 'cliente': {'genero': 'feminino', 'idoso': 0, 'parceiro': 'sim', 'dependentes': 'sim', 'tempo_servico': 9}, 'telefone': {'servico_telefone': 'sim', 'varias_linhas': 'nao'}, 'internet': {'servico_internet': 'DSL', 'seguranca_online': 'nao', 'backup_online': 'sim', 'protecao_dispositivo': 'nao', 'suporte_tecnico': 'sim', 'tv_streaming': 'sim', 'filmes_streaming': 'nao'}, 'conta': {'contrato': None, 'faturamente_eletronico': None, 'metodo_pagamento': None, 'cobranca': {'mensal': None, 'Total': None}}}, {'id_cliente': '0003-MKNFE', 'Churn': 'nao', 'cliente': {'genero': 'masculino', 'idoso': 0, 'parceiro': 'nao', 'dependentes': 'nao', 'tempo_servico': 9}, 'telefone': {'servico_telefone': 'sim', 'varias_linhas': 'sim'}, 'internet': {'servico_internet': 'DSL', 'seguranca_online': 'nao', 'backup_online': 'nao', 'protecao_dispositivo': 'nao', 'suporte_tecnico': 'nao', 'tv_streaming': 'nao', 'filmes_streaming': 'sim'}, 'conta': {'contrato': 'mes a me

>A estrutura acima foi parcialmente transcrita. Para conferi-la na íntegra, execute o código na sua máquina.


Na próxima célula, digitamos pd.json_normalize(), mas em vez de passarmos uma coluna do DataFrame, passamos json_bruto

In [8]:
dados_normalizados = pd.json_normalize(json_bruto)
dados_normalizados.head()

,id_cliente,Churn,cliente.genero,cliente.idoso,cliente.parceiro,cliente.dependentes,cliente.tempo_servico,telefone.servico_telefone,telefone.varias_linhas,internet.servico_internet,...,internet.backup_online,internet.protecao_dispositivo,internet.suporte_tecnico,internet.tv_streaming,internet.filmes_streaming,conta.contrato,conta.faturamente_eletronico,conta.metodo_pagamento,conta.cobranca.mensal,conta.cobranca.Total
0,0002-ORFBO,nao,feminino,0,sim,sim,9.0,sim,nao,DSL,...,sim,nao,sim,sim,nao,None,None,None,NaN,None
1,0003-MKNFE,nao,masculino,0,nao,nao,9.0,sim,sim,DSL,...,nao,nao,nao,nao,sim,mes a mes,nao,cheque pelo correio,59.9,542.4
2,0004-TLHLJ,sim,masculino,0,nao,nao,4.0,sim,nao,fibra otica,...,nao,sim,nao,nao,nao,mes a mes,sim,cheque eletronico,73.9,280.85
3,0011-IGKFF,sim,masculino,1,sim,nao,13.0,sim,nao,fibra otica,...,sim,sim,nao,sim,sim,mes a mes,sim,cheque eletronico,98.0,1237.85
4,0013-EXCHZ,sim,feminino,1,sim,nao,3.0,sim,nao,fibra otica,...,nao,nao,sim,sim,nao,mes a mes,sim,cheque pelo correio,83.9,267.4


Ao analisarmos os dados normalizados, notamos que as primeiras colunas são as mesmas do objeto JSON original, como id_cliente e Churn. Porém, a partir da terceira coluna, há diferenças significativas. Por exemplo, a coluna cliente.genero era uma chave do JSON de cliente original que continha outro JSON dentro dela.

Aqui está uma versão mais **fluida, informativa e atraente** do seu texto, mantendo o conteúdo técnico e detalhado:

---

### Para saber mais: parâmetros do `json_normalize`

Próxima Atividade

O método `json_normalize()` do Pandas é usado para transformar dados JSON em um **formato tabular**, como DataFrames, tornando-os muito mais fáceis de analisar e manipular.

Os principais parâmetros são:

* **data**: o objeto JSON a ser normalizado.
* **record\_path**: caminho para acessar arrays de registros dentro do JSON.
* **meta**: lista de colunas adicionais a serem incluídas no DataFrame, além das colunas normalizadas.
* **errors**: define como lidar com erros (`"raise"` para lançar erro ou `"ignore"` para ignorar).
* **sep**: separador usado para concatenar chaves de objetos aninhados (padrão: `"."`).

#### Exemplo prático:

```python
import pandas as pd

data = {
    "empresa": "alura",
    "funcionarios": [
        {"nome": "Alice", "endereço": {"cidade": "São Paulo", "estado": "SP"}},
        {"nome": "Bob", "endereço": {"cidade": "Rio de Janeiro", "estado": "RJ"}}
    ]
}

df = pd.json_normalize(data, record_path='funcionarios', meta='empresa', errors='ignore')
print(df)
```

**Saída:**

| nome  | endereço.cidade | endereço.estado | empresa |
| ----- | --------------- | --------------- | ------- |
| Alice | São Paulo       | SP              | alura   |
| Bob   | Rio de Janeiro  | RJ              | alura   |

#### Como funciona:

O objeto `data` contém a chave `"empresa"` e uma lista de `"funcionarios"`. Cada funcionário é representado por um dicionário com informações de nome e endereço (que é um dicionário aninhado com cidade e estado).

O `json_normalize()` transforma essas informações em linhas do DataFrame, usando `record_path='funcionarios'` para normalizar os dados da lista, `meta='empresa'` para incluir a empresa em cada linha e `errors='ignore'` para evitar que erros interrompam o processo.

O resultado é uma tabela clara e organizada, com **cada funcionário em uma linha** e seus dados em colunas separadas — perfeita para análise ou processamento posterior.

---

#Desafio: utilizando parâmetros do json_normalize

Ao normalizar o objeto JSON durante a aula passamos simplesmente pd.json_normalize(<dados>) mas o método json_normalize possui diversos parâmetros para trabalhar com estruturas de dados mais complexas.

Você recebeu a tarefa de normalizar o seguinte arquivo JSON chamado “informacoes.json” referente a algumas informações de identificação de um cliente. O arquivo possui o seguinte conteúdo:

In [9]:
dados = {
  "nome": "João",
  "idade": 28,
  "enderecos": [
    {
      "tipo": "casa",
      "rua": "Rua A",
      "numero": 123,
      "cidade": "São Paulo"
    },
    {
      "tipo": "trabalho",
      "rua": "Rua B",
      "numero": 456,
      "cidade": "Rio de Janeiro"
    }
  ]
}

import json

with open('informacoes.json', 'w') as f:
    json.dump(dados, f, indent=4) #indent=4: Ident = opcional, para aninhar a saida(4 espaços).

print("Dados salvos em informacoes.json")

Dados salvos em informacoes.json


In [10]:
pd.read_json('informacoes.json')

,nome,idade,enderecos
0,João,28,"{'tipo': 'casa', 'rua': 'Rua A', 'numero': 123..."
1,João,28,"{'tipo': 'trabalho', 'rua': 'Rua B', 'numero':..."


In [11]:
# record_path define qual caminho deve ser seguido para obter os dados que
# serão normalizados
# O parâmetro meta é usado para especificar quais chaves do objeto original
# queremos incluir na saída normalizada

informacoes = pd.json_normalize(dados, record_path='enderecos', meta=['nome', 'idade'])
informacoes

,tipo,rua,numero,cidade,nome,idade
0,casa,Rua A,123,São Paulo,João,28
1,trabalho,Rua B,456,Rio de Janeiro,João,28
